## 1. Đọc dữ liệu từ file CSV

In [1]:
# Import SparkSession
from pyspark.sql import SparkSession

# Khởi tạo SparkSession
spark = SparkSession.builder \
    .appName("Fecom_Inc_Analysis") \
    .getOrCreate()

# Đọc dữ liệu từ các file CSV vào Spark DataFrames
orders_df = spark.read.format("csv").options(header='True', delimiter=';', inferSchema='True').load("Orders.csv")
customers_df = spark.read.format("csv").options(header='True', delimiter=';', inferSchema='True').load("Customer_List.csv")
products_df = spark.read.format("csv").options(header='True', delimiter=';', inferSchema='True').load("Products.csv")
order_items_df = spark.read.format("csv").options(header='True', delimiter=';', inferSchema='True').load("Order_Items.csv")
reviews_df = spark.read.format("csv").options(header='True', delimiter=';', inferSchema='True').load("Order_Reviews.csv")

# In schema của từng DataFrame để hiểu cấu trúc của chúng
orders_df.printSchema()
customers_df.printSchema()
products_df.printSchema()
order_items_df.printSchema()
reviews_df.printSchema()

root
 |-- Order_ID: string (nullable = true)
 |-- Customer_Trx_ID: string (nullable = true)
 |-- Order_Status: string (nullable = true)
 |-- Order_Purchase_Timestamp: timestamp (nullable = true)
 |-- Order_Approved_At: timestamp (nullable = true)
 |-- Order_Delivered_Carrier_Date: timestamp (nullable = true)
 |-- Order_Delivered_Customer_Date: timestamp (nullable = true)
 |-- Order_Estimated_Delivery_Date: timestamp (nullable = true)

root
 |-- Customer_Trx_ID: string (nullable = true)
 |-- Subscriber_ID: string (nullable = true)
 |-- Subscribe_Date: date (nullable = true)
 |-- First_Order_Date: date (nullable = true)
 |-- Customer_Postal_Code: string (nullable = true)
 |-- Customer_City: string (nullable = true)
 |-- Customer_Country: string (nullable = true)
 |-- Customer_Country_Code: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)

root
 |-- Product_ID: string (nullable = true)
 |-- Product_Category_Name: string (nullable = true)
 

In [2]:
# Hiển thị một số dòng dữ liệu của các Dataframe
customers_df.show(10)
products_df.show(10)
orders_df.show(10)
order_items_df.show(10)
reviews_df.show(10)

+--------------------+--------------------+--------------+----------------+--------------------+-------------+----------------+---------------------+---+------+
|     Customer_Trx_ID|       Subscriber_ID|Subscribe_Date|First_Order_Date|Customer_Postal_Code|Customer_City|Customer_Country|Customer_Country_Code|Age|Gender|
+--------------------+--------------------+--------------+----------------+--------------------+-------------+----------------+---------------------+---+------+
|1e959e1f5920cba43...|9765e039028279fd2...|    2023-07-08|      2023-07-09|            FR-75005|        Paris|          France|                   FR| 29|  Male|
|9877437582f263da7...|a75e134e7eb6f96e2...|    2024-03-23|      2024-04-11|           PL-00-001|       Warsaw|          Poland|                   PL| 38|  Male|
|fa6fbbb2080646aca...|2fdac27295500e820...|    2023-05-12|      2023-06-01|             NL-1012|    Amsterdam|     Netherlands|                   NL| 35|Female|
|a4c9ff14ae7620126...|e9ab8fd8ea96

## 2. Thống kê tổng số đơn hàng, số lượng khách hàng và người bán

In [3]:
# Tổng số đơn hàng
total_orders = orders_df.count()
print("Tổng số đơn hàng:", total_orders)

# Tổng số khách hàng
total_customers = customers_df.select("Customer_Trx_ID").distinct().count()
print("Tổng số khách hàng:", total_customers)

# Tổng số người bán
total_sellers = order_items_df.select("Seller_ID").distinct().count()
print("Tổng số người bán:", total_sellers)

Tổng số đơn hàng: 99441
Tổng số khách hàng: 99442
Tổng số người bán: 3095


## 3. Phân tích số lượng đơn hàng theo quốc gia, sắp xếp theo thứ tự giảm dần

In [4]:
# Join Orders và Customer_List theo Customer_Trx_ID
orders_customers_df = orders_df.join(customers_df, on="Customer_Trx_ID", how="inner")

# Thống kê số đơn hàng theo quốc gia
orders_by_country = orders_customers_df.groupBy("Customer_Country").count().orderBy("count", ascending=False)
orders_by_country.show()

+----------------+-----+
|Customer_Country|count|
+----------------+-----+
|         Germany|41754|
|          France|12848|
|     Netherlands|11629|
|         Belgium| 5464|
|         Austria| 5043|
|     Switzerland| 3640|
|  United Kingdom| 3382|
|          Poland| 2139|
|         Czechia| 2034|
|           Italy| 2025|
|           Spain| 1651|
|        Portugal| 1336|
|          Sweden|  975|
|         Denmark|  905|
|          Serbia|  746|
|          Norway|  716|
|        Slovakia|  534|
|        Slovenia|  495|
|          Turkey|  485|
|          Greece|  412|
+----------------+-----+
only showing top 20 rows



## 4. Phân tích số lượng đơn hàng nhóm theo năm, tháng đặt hàng

In [5]:
from pyspark.sql.functions import year, month

# Phân nhóm theo năm và tháng của Order_Purchase_Timestamp
orders_by_time = orders_df.withColumn("year", year("Order_Purchase_Timestamp")) \
                          .withColumn("month", month("Order_Purchase_Timestamp"))

orders_time_summary = orders_by_time.groupBy("year", "month").count().sort("year", "month", ascending=[True, False])
orders_time_summary.show()

+----+-----+-----+
|year|month|count|
+----+-----+-----+
|2022|   12|    1|
|2022|   10|  324|
|2022|    9|    4|
|2023|   12| 5673|
|2023|   11| 7544|
|2023|   10| 4631|
|2023|    9| 4285|
|2023|    8| 4331|
|2023|    7| 4026|
|2023|    6| 3245|
|2023|    5| 3700|
|2023|    4| 2404|
|2023|    3| 2682|
|2023|    2| 1780|
|2023|    1|  800|
|2024|   10|    4|
|2024|    9|   16|
|2024|    8| 6512|
|2024|    7| 6292|
|2024|    6| 6167|
+----+-----+-----+
only showing top 20 rows



## 5. Thống kê điểm đánh giá trung bình, số lượng đánh giá theo từng mức

In [6]:
reviews_df.groupBy("Review_Score").count().orderBy("Review_Score").show()

+----------------+-----+
|    Review_Score|count|
+----------------+-----+
|            NULL|   45|
|               1|11424|
|               2| 3151|
|2024-04-07 04:19|    1|
|2024-07-05 11:11|    1|
|               3| 8179|
|               4|19141|
|               5|57328|
+----------------+-----+



In [7]:
from pyspark.sql.functions import col

reviews_df = reviews_df.withColumn("Review_Score", col("Review_Score").cast("int"))
reviews_df.printSchema()

root
 |-- Review_ID: string (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Review_Score: integer (nullable = true)
 |-- Review_Comment_Title_En: string (nullable = true)
 |-- Review_Comment_Message_En: string (nullable = true)
 |-- Review_Creation_Date: string (nullable = true)
 |-- Review_Answer_Timestamp: timestamp (nullable = true)



In [8]:
reviews_df = reviews_df.filter(~col("Review_Score").cast("int").isNull())
reviews_df = reviews_df.withColumn("Review_Score", col("Review_Score").cast("int"))

# Thống kê điểm đánh giá trung bình
avg_review = reviews_df.groupBy().avg("Review_Score").first()[0]
print("Điểm đánh giá trung bình:", avg_review)

# Đếm số lượng đánh giá theo từng mức
reviews_by_score = reviews_df.groupBy("Review_Score").count().orderBy("Review_Score")
reviews_by_score.show()

Điểm đánh giá trung bình: 4.0864214950162765
+------------+-----+
|Review_Score|count|
+------------+-----+
|           1|11424|
|           2| 3151|
|           3| 8179|
|           4|19141|
|           5|57328|
+------------+-----+



## 6. Tính doanh thu (giá sản phẩm + phí vận chuyển) trong năm 2024 và nhóm theo danh mục sản phẩm

In [9]:
from pyspark.sql.functions import to_timestamp, year, col, sum as _sum

# Chuyển đổi cột Order_Purchase_Timestamp
orders_df = orders_df.withColumn("Order_Purchase_Timestamp", to_timestamp("Order_Purchase_Timestamp", "yyyy-MM-dd HH:mm:ss"))

# Lọc Order trong năm 2024
orders_2024 = orders_df.filter(year("Order_Purchase_Timestamp") == 2024)

# Join Orders với Order_Items
order_item_2024 = order_items_df.join(
    orders_2024.select("Order_ID"),
    on="Order_ID",
    how="inner"
)

# Join tiếp với Products để lấy danh mục
item_product_2024 = order_item_2024.join(
    products_df.select("Product_ID", "Product_Category_Name"),
    on="Product_ID",
    how="inner"
)

# Tính doanh thu (Price + Freight_Value)
item_revenue = item_product_2024.withColumn(
    "Total_Price",
    col("Price") + col("Freight_Value")
)

# Nhóm theo danh mục sản phẩm
revenue_by_category = item_revenue.groupBy("Product_Category_Name") \
    .agg(_sum("Total_Price").alias("Revenue")) \
    .orderBy(col("Revenue").desc())

revenue_by_category.show(50, truncate=False)

+---------------------------------------+------------------+
|Product_Category_Name                  |Revenue           |
+---------------------------------------+------------------+
|Health_Beauty                          |885191.119999997  |
|Watches_Gifts                          |771986.750000001  |
|Bed_Bath_Table                         |650794.700000002  |
|Sports_Leisure                         |621999.3399999994 |
|Computers_Accessories                  |594771.0400000002 |
|Housewares                             |491576.9600000012 |
|Furniture_Decor                        |476466.1300000007 |
|Auto                                   |404210.5700000002 |
|Baby                                   |299052.5599999998 |
|Cool_Stuff                             |273910.0500000001 |
|Garden_Tools                           |259068.31999999983|
|Telephony                              |217452.1299999995 |
|Perfumery                              |204562.53999999992|
|Toys                   